In [32]:
from blackjack import Game
from tqdm import tqdm
from collections import defaultdict
import random
import numpy as np

In [ ]:
rng = np.random.default_rng(42)
random.seed(42)

In [34]:
def get_state(game: Game):
    player_card_ranks = []
    for card in game.player_hand:
        player_card_ranks.append(min(card.value, 10))

    state = (
        tuple(sorted(player_card_ranks)),
        min(game.dealer_hand[0].value, 10)
    )

    return state

In [47]:
# Q(s, 0) is stay and Q(s, 1) is hit.
ACTIONS = np.array(["S", "H"])
q_table = defaultdict(lambda: np.zeros(len(ACTIONS), dtype=np.float64))

n_iter = 1_000_000
epsilon = 1.0
min_epsilon = 0.01
epsilon_decay = (min_epsilon / epsilon) ** (1 / n_iter)
alpha = 0.1
gamma = 0.99

In [ ]:
training_outcomes = np.zeros(3, dtype=np.int64)  # losses, draws, wins

for _ in tqdm(range(n_iter)):
    game = Game(five_card_charlie=True)

    while not game.is_over:
        state = get_state(game)

        if rng.random() < epsilon:
            action_index = int(rng.integers(len(ACTIONS)))
        else:
            values = q_table[state]
            best_actions = np.flatnonzero(values == values.max())
            action_index = int(rng.choice(best_actions))

        game.take_action(ACTIONS[action_index])

        if game.is_over:
            target_q_value = game.score
        else:
            next_state = get_state(game)
            
            # current reward is 0, so bellman's equation simplifies to gamma * max Q(s', a')
            target_q_value = gamma * np.max(q_table[next_state])

        current = q_table[state][action_index]
        q_table[state][action_index] += alpha * (target_q_value - current)

    training_outcomes[game.score + 1] += 1
    epsilon = max(epsilon * epsilon_decay, min_epsilon)

print(f"Learned {len(q_table):,} states.")
print(dict(zip(("losses", "draws", "wins"), training_outcomes)))

100%|██████████| 1000000/1000000 [00:56<00:00, 17771.93it/s]

Learned 4,915 states.
{'losses': np.int64(518183), 'draws': np.int64(69200), 'wins': np.int64(412617)}


In [49]:
def choose_greedy_action(game: Game) -> str:
    """Choose a learned action, breaking ties randomly."""
    values = q_table.get(get_state(game))
    if values is None:
        return str(rng.choice(ACTIONS))

    return str(ACTIONS[int(np.argmax(values))])


def evaluate_agent(n_games: int = 10_000) -> dict[str, float]:
    outcomes = np.zeros(3, dtype=np.int64)

    for _ in range(n_games):
        game = Game(five_card_charlie=True)
        while not game.is_over:
            game.take_action(choose_greedy_action(game))
        outcomes[game.score + 1] += 1

    return {
        "loss_rate": outcomes[0] / n_games,
        "draw_rate": outcomes[1] / n_games,
        "win_rate": outcomes[2] / n_games,
        "average_reward": (outcomes[2] - outcomes[0]) / n_games,
    }


evaluate_agent()

{'loss_rate': np.float64(0.4871),
 'draw_rate': np.float64(0.0804),
 'win_rate': np.float64(0.4325),
 'average_reward': np.float64(-0.0546)}

In [50]:
q_table[((6, 10), 3)]

array([-0.72681607, -0.52423424])